# 03. Causal Problem Formulation

## EconoCausal: AI-Powered Personalized Marketing Campaign Optimization

### Objective

This notebook formally defines the causal inference problem for the EconoCausal project.

Notebook 01 established the structure of the Hillstrom Email Marketing dataset, while Notebook 02 created the reusable preprocessing pipeline and isolated the causal components:

- Covariates ($X$)
- Treatment ($T$)
- Outcomes ($Y$)

The objective of this notebook is to translate the business problem into a formal causal framework that can be used by the downstream Propensity Score Modeling and Double Machine Learning notebooks.

### Main Questions

1. What is the causal effect of sending a marketing email?
2. How does the effect differ between Mens E-Mail and Womens E-Mail?
3. Which customer outcomes should be optimized?
4. Which pre-treatment customer characteristics should be considered during causal estimation?
5. What causal assumptions are required to identify the treatment effect?

In [1]:

import logging
from pathlib import Path

import numpy as np
import pandas as pd

from dowhy import CausalModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger(__name__)

logger.info("Libraries imported successfully.")

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-25 15:46:31 [INFO] Libraries imported successfully.


## Causal Setup

The Hillstrom dataset is a randomized email marketing experiment.

### Treatment

The treatment variable is `segment`.

It contains three campaign groups:

- `No E-Mail` → Control group
- `Mens E-Mail` → Mens campaign treatment
- `Womens E-Mail` → Womens campaign treatment

### Outcomes

The causal outcomes are:

- `visit`
- `conversion`
- `spend`

The objective is to estimate how assigning a customer to a marketing campaign changes these outcomes compared with the control condition.

In [2]:
DATA_PATHS = [
    Path("../Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
    Path("Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
    Path("../datasets/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
]

dataset_path = next((path for path in DATA_PATHS if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Hillstrom dataset CSV not found. Please verify the project directory."
    )

df = pd.read_csv(dataset_path)

df = df.drop_duplicates().reset_index(drop=True)

logger.info(f"Dataset loaded successfully: {df.shape}")
logger.info(f"Remaining duplicate rows: {df.duplicated().sum()}")

2026-09-25 15:48:14 [INFO] Dataset loaded successfully: (57438, 12)
2026-09-25 15:48:14 [INFO] Remaining duplicate rows: 0


In [3]:
TREATMENT_COL = "segment"

OUTCOME_COLS = [
    "visit",
    "conversion",
    "spend"
]

COVARIATE_COLS = [
    "recency",
    "history_segment",
    "history",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel"
]

logger.info(f"Treatment variable: {TREATMENT_COL}")
logger.info(f"Outcome variables: {OUTCOME_COLS}")
logger.info(f"Covariates: {COVARIATE_COLS}")

2026-09-25 15:48:42 [INFO] Treatment variable: segment
2026-09-25 15:48:42 [INFO] Outcome variables: ['visit', 'conversion', 'spend']
2026-09-25 15:48:42 [INFO] Covariates: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel']


In [4]:
TREATMENT_MAPPING = {
    "No E-Mail": 0,
    "Mens E-Mail": 1,
    "Womens E-Mail": 2
}

df["treatment"] = df[TREATMENT_COL].map(TREATMENT_MAPPING)

if df["treatment"].isnull().any():
    unknown_values = df.loc[
        df["treatment"].isnull(),
        TREATMENT_COL
    ].unique()

    raise ValueError(
        f"Unknown treatment categories found: {unknown_values}"
    )

treatment_labels = {
    0: "No E-Mail",
    1: "Mens E-Mail",
    2: "Womens E-Mail"
}

treatment_summary = (
    df["treatment"]
    .value_counts()
    .sort_index()
    .rename(index=treatment_labels)
    .to_frame("customer_count")
)

treatment_summary["percentage"] = (
    treatment_summary["customer_count"] / len(df) * 100
).round(2)

treatment_summary

,customer_count,percentage
treatment,,
No E-Mail,19081,33.22
Mens E-Mail,19183,33.40
Womens E-Mail,19174,33.38


## Potential Outcomes Framework

For each customer $i$, let:

$$
Y_i(t)
$$

represent the potential outcome that would be observed if customer $i$ received treatment $t$.

The treatment values are:

- $t=0$: No E-Mail
- $t=1$: Mens E-Mail
- $t=2$: Womens E-Mail

For example, the individual treatment effect of the Mens campaign compared with control is:

$$
\tau_i^{Mens}
=
Y_i(1) - Y_i(0)
$$

Similarly, the individual treatment effect of the Womens campaign compared with control is:

$$
\tau_i^{Womens}
=
Y_i(2) - Y_i(0)
$$

Because a customer can receive only one treatment in the experiment, the unobserved potential outcomes must be estimated using causal inference methods.

## Causal Questions and Estimands

The main causal questions are:

1. What is the causal effect of the Mens E-Mail campaign compared with No E-Mail?
2. What is the causal effect of the Womens E-Mail campaign compared with No E-Mail?

These effects are evaluated for three outcomes:

- Visit
- Conversion
- Spend

### Average Treatment Effect

For a treatment $t$ compared with control:

$$
ATE_t =
E[Y(t) - Y(0)]
$$

The downstream modeling stage will additionally estimate heterogeneous treatment effects so that different customer groups can receive different campaign recommendations.

## Identification Assumptions

The Hillstrom dataset originates from a randomized controlled experiment.

The important causal assumptions are:

### 1. Randomized Treatment Assignment

Customers were randomly assigned to the campaign groups.

Therefore, treatment assignment is not expected to systematically depend on the observed customer characteristics.

### 2. Positivity

Each relevant customer profile should have a non-zero probability of receiving the treatment being compared.

### 3. Consistency

The observed outcome under the received treatment corresponds to the potential outcome for that treatment.

### 4. No Interference

One customer's treatment assignment should not directly change another customer's outcome.

### 5. Pre-treatment Covariates

The customer characteristics used for causal analysis should be measured before the marketing treatment.

Because this dataset is randomized, the covariates are not required to make the original treatment assignment unconfounded. They are still useful for heterogeneous treatment-effect modeling and for maintaining an architecture that can later be adapted to observational data.

In [5]:
causal_graph = """
digraph {
    recency -> visit;
    recency -> conversion;
    recency -> spend;

    history_segment -> visit;
    history_segment -> conversion;
    history_segment -> spend;

    history -> visit;
    history -> conversion;
    history -> spend;

    mens -> visit;
    mens -> conversion;
    mens -> spend;

    womens -> visit;
    womens -> conversion;
    womens -> spend;

    zip_code -> visit;
    zip_code -> conversion;
    zip_code -> spend;

    newbie -> visit;
    newbie -> conversion;
    newbie -> spend;

    channel -> visit;
    channel -> conversion;
    channel -> spend;

    treatment -> visit;
    treatment -> conversion;
    treatment -> spend;
}
"""

print(causal_graph)


digraph {
    recency -> visit;
    recency -> conversion;
    recency -> spend;

    history_segment -> visit;
    history_segment -> conversion;
    history_segment -> spend;

    history -> visit;
    history -> conversion;
    history -> spend;

    mens -> visit;
    mens -> conversion;
    mens -> spend;

    womens -> visit;
    womens -> conversion;
    womens -> spend;

    zip_code -> visit;
    zip_code -> conversion;
    zip_code -> spend;

    newbie -> visit;
    newbie -> conversion;
    newbie -> spend;

    channel -> visit;
    channel -> conversion;
    channel -> spend;

    treatment -> visit;
    treatment -> conversion;
    treatment -> spend;
}



## Structural Causal Graph

The DAG represents the assumed causal structure.

Customer characteristics such as:

- recency
- historical spending
- history segment
- gender/product affinity indicators
- zip code
- new-customer status
- channel

are connected to the customer outcomes.

The treatment assignment is connected to the outcomes because the purpose of the analysis is to estimate the causal impact of the marketing campaign.

Since the Hillstrom dataset is based on randomized treatment assignment, direct covariate-to-treatment confounding is not required for the original experimental identification.

In [6]:
causal_data = df[
    COVARIATE_COLS + ["treatment"] + OUTCOME_COLS
].copy()

identification_results = {}

for outcome in OUTCOME_COLS:
    logger.info(
        f"Running causal identification for outcome: {outcome}"
    )

    model = CausalModel(
        data=causal_data,
        treatment="treatment",
        outcome=outcome,
        graph=causal_graph
    )

    identified_estimand = model.identify_effect(
        proceed_when_unidentifiable=True
    )

    identification_results[outcome] = identified_estimand

    print("=" * 80)
    print(f"Outcome: {outcome}")
    print(identified_estimand)

2026-09-25 15:50:59 [INFO] Running causal identification for outcome: visit
2026-09-25 15:50:59 [ERROR] Error: Pygraphviz cannot be loaded. No module named 'pygraphviz'
Trying pydot ...
2026-09-25 15:50:59 [INFO] Model to find the causal effect of treatment ['treatment'] on outcome ['visit']
2026-09-25 15:50:59 [INFO] Causal effect can be identified.
2026-09-25 15:50:59 [INFO] Instrumental variables for treatment and outcome:[]
2026-09-25 15:50:59 [INFO] Frontdoor variables for treatment and outcome:[]
2026-09-25 15:50:59 [INFO] Number of general adjustment sets found: 1
2026-09-25 15:50:59 [INFO] Causal effect can be identified.
2026-09-25 15:50:59 [INFO] Running causal identification for outcome: conversion
2026-09-25 15:50:59 [ERROR] Error: Pygraphviz cannot be loaded. No module named 'pygraphviz'
Trying pydot ...
2026-09-25 15:50:59 [INFO] Model to find the causal effect of treatment ['treatment'] on outcome ['conversion']
2026-09-25 15:50:59 [INFO] Causal effect can be identified.

Outcome: visit
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                
────────────(E[visit])
d[treatment]          
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→visit then P(visit|treatment,,U) = P(visit|treatment,)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
     d                
────────────(E[visit])
d[treatment]          
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→visit then P(visit|treatment,,U) = P(visit|treatment,)

Outcome: conversion
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                     
────────────(E[conversion])
d[treatment]               
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→conversion then 

2026-09-25 15:50:59 [INFO] Model to find the causal effect of treatment ['treatment'] on outcome ['spend']
2026-09-25 15:50:59 [INFO] Causal effect can be identified.
2026-09-25 15:50:59 [INFO] Instrumental variables for treatment and outcome:[]
2026-09-25 15:50:59 [INFO] Frontdoor variables for treatment and outcome:[]
2026-09-25 15:50:59 [INFO] Number of general adjustment sets found: 1
2026-09-25 15:50:59 [INFO] Causal effect can be identified.


Outcome: spend
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                
────────────(E[spend])
d[treatment]          
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→spend then P(spend|treatment,,U) = P(spend|treatment,)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
     d                
────────────(E[spend])
d[treatment]          
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→spend then P(spend|treatment,,U) = P(spend|treatment,)



## Backdoor Adjustment Perspective

The causal graph provides a formal representation of the variables that may explain customer outcomes.

For observational causal inference, backdoor adjustment would require controlling for appropriate pre-treatment variables that block non-causal paths between treatment and outcome.

In this project, the original Hillstrom data comes from a randomized experiment, so treatment assignment is already designed to remove systematic confounding.

The covariates remain important for:

- modeling heterogeneous treatment effects,
- understanding customer-level response,
- building reusable causal pipelines,
- and extending the framework to future observational datasets.

In [7]:
mens_data = causal_data[
    causal_data["treatment"].isin([0, 1])
].copy()

mens_data["binary_treatment"] = (
    mens_data["treatment"] == 1
).astype(int)

womens_data = causal_data[
    causal_data["treatment"].isin([0, 2])
].copy()

womens_data["binary_treatment"] = (
    womens_data["treatment"] == 2
).astype(int)

logger.info(
    f"Mens vs Control dataset shape: {mens_data.shape}"
)

logger.info(
    f"Womens vs Control dataset shape: {womens_data.shape}"
)

logger.info(
    "Pairwise treatment datasets prepared for downstream modeling."
)

2026-09-25 15:51:37 [INFO] Mens vs Control dataset shape: (38264, 13)
2026-09-25 15:51:37 [INFO] Womens vs Control dataset shape: (38255, 13)
2026-09-25 15:51:37 [INFO] Pairwise treatment datasets prepared for downstream modeling.


## Pairwise Treatment Comparisons

The multi-arm campaign assignment is converted into two binary treatment comparisons for downstream causal modeling.

### Comparison 1

Mens E-Mail vs No E-Mail

- `0` → No E-Mail
- `1` → Mens E-Mail

### Comparison 2

Womens E-Mail vs No E-Mail

- `0` → No E-Mail
- `1` → Womens E-Mail

These pairwise datasets allow the downstream propensity-score and Double Machine Learning stages to estimate campaign-specific treatment effects.

In [8]:
causal_specification = {
    "treatment": {
        "variable": "treatment",
        "control": "No E-Mail",
        "treatments": [
            "Mens E-Mail",
            "Womens E-Mail"
        ]
    },
    "outcomes": OUTCOME_COLS,
    "covariates": COVARIATE_COLS,
    "estimands": [
        "Mens E-Mail vs No E-Mail",
        "Womens E-Mail vs No E-Mail"
    ],
    "primary_objective": "Estimate heterogeneous incremental treatment effects",
    "identification_design": "Randomized controlled experiment"
}

causal_specification

{'treatment': {'variable': 'treatment',
  'control': 'No E-Mail',
  'treatments': ['Mens E-Mail', 'Womens E-Mail']},
 'outcomes': ['visit', 'conversion', 'spend'],
 'covariates': ['recency',
  'history_segment',
  'history',
  'mens',
  'womens',
  'zip_code',
  'newbie',
  'channel'],
 'estimands': ['Mens E-Mail vs No E-Mail', 'Womens E-Mail vs No E-Mail'],
 'primary_objective': 'Estimate heterogeneous incremental treatment effects',
 'identification_design': 'Randomized controlled experiment'}

## Conclusion

The causal problem has now been formally specified.

### Defined

- Treatment variable: `segment`
- Encoded treatment: `0, 1, 2`
- Control group: `No E-Mail`
- Treatment groups:
  - `Mens E-Mail`
  - `Womens E-Mail`
- Outcomes:
  - `visit`
  - `conversion`
  - `spend`
- Customer covariates
- Potential outcomes
- Causal estimands
- Identification assumptions
- Structural Causal Graph
- DoWhy causal identification
- Pairwise treatment datasets

### Next Step

The next notebooks will use this causal formulation for:

1. Propensity score modeling
2. Double Machine Learning
3. Treatment-effect estimation
4. Customer-level heterogeneous treatment effects
5. Campaign recommendation and optimization

This notebook therefore establishes the causal foundation for the EconoCausal pipeline.